## GeoProxy data pipeline → `sites.json` + `targets.json`

One notebook for **all bodies**. Earth analog sites and Moon/Mars reference targets go through **the same code**, so their numbers are directly comparable.

**Run:** `Runtime → Run all` (about 2–4 minutes; the Moon south pole is the slowest).

**Cells:**
1. Install · 2. **Settings** (the only cell you normally edit) · 3. Data sources · 4. Core functions · 5. **Self-test** (no internet needed) · 6. Data readers · 7. **Connection test** · 8. Measure everything · 9. Maps · 10. Build JSON · 11. Save

**If something fails:** the cell number tells you where. Cell 5 = maths problem, cell 7 = network or data-source problem, cell 8 = a specific site (the run continues for the other sites and prints the error). To debug one site, put its id in `ONLY` in cell 2 and run all again.

### 1 · Install
*Why:* `rasterio` reads geographic raster files (GeoTIFF). Everything else (numpy, scipy, pandas, requests, matplotlib) is already in Colab.

In [ ]:
!pip -q install rasterio

### 2 · Settings: the only cell you normally edit
- **SITES** = Earth places we test as analogs. **TARGETS** = Moon/Mars reference places.
- `lat`/`lon` in decimal degrees; south and west are negative; Mars/Moon longitudes are east-positive (−180…180).
- `analog_for` and `notes` record **your team's hypothesis**. They are not measurements.
- `LITERATURE` holds values taken from published papers (e.g. Moon/Mars temperatures), **always with a citation**. Leave a value out rather than guess: the app treats it as missing.
- `ONLY = ["mount-cameroon"]` runs just that location, which is handy for debugging.

In [ ]:
SITES = [
    {"id": "mount-cameroon", "name": "Mount Cameroon", "region": "Southwest Region, Cameroon", "body": "earth",
     "lat": 4.217, "lon": 9.167, "analog_for": ["mars"],
     "notes": "Active basaltic volcano (summit area). Team hypothesis: Martian volcanic terrain."},
    {"id": "atacama-yungay", "name": "Yungay, Atacama Desert", "region": "Antofagasta, Chile", "body": "earth",
     "lat": -24.08, "lon": -70.0, "analog_for": ["mars"],
     "notes": "Hyper-arid desert. Team hypothesis: dry Martian surface."},
    {"id": "dry-valleys-vanda", "name": "Wright Valley (Lake Vanda)", "region": "McMurdo Dry Valleys, Antarctica", "body": "earth",
     "lat": -77.53, "lon": 161.58, "analog_for": ["mars", "moon"],
     "notes": "Cold polar desert. Team hypothesis: cold, dry Mars and polar Moon."},
    {"id": "haughton-crater", "name": "Haughton impact crater", "region": "Devon Island, Nunavut, Canada", "body": "earth",
     "lat": 75.367, "lon": -89.683, "analog_for": ["mars", "moon"],
     "notes": "Polar-desert impact crater. Team hypothesis: cratered terrain."},
    {"id": "mauna-kea", "name": "Mauna Kea summit", "region": "Hawaii, USA", "body": "earth",
     "lat": 19.821, "lon": -155.468, "analog_for": ["moon", "mars"],
     "notes": "High-altitude basaltic volcano. Team hypothesis: volcanic regolith."},
    # --- Apollo-era astronaut geology training grounds (Moon) ---
    {"id": "meteor-crater", "name": "Meteor Crater (Barringer)", "region": "Arizona, USA", "body": "earth",
     "lat": 35.027, "lon": -111.022, "analog_for": ["moon", "mars"],
     "notes": "Simple impact crater about 1.2 km wide; Apollo astronaut training site. Crater is small compared with the 20 km window."},
    {"id": "ries-crater", "name": "Nördlinger Ries", "region": "Bavaria, Germany", "body": "earth",
     "lat": 48.88, "lon": 10.56, "analog_for": ["moon"],
     "notes": "About 24 km complex impact crater; Apollo 14 crew geology training site."},
    {"id": "askja-iceland", "name": "Askja caldera", "region": "Highlands, Iceland", "body": "earth",
     "lat": 65.05, "lon": -16.75, "analog_for": ["moon", "mars"],
     "notes": "Basaltic volcanic highland; Apollo astronaut training site (1965, 1967)."},
    {"id": "lonar-crater", "name": "Lonar crater", "region": "Maharashtra, India", "body": "earth",
     "lat": 19.976, "lon": 76.508, "analog_for": ["moon", "mars"],
     "notes": "Impact crater in basalt; compared with craters in lunar and Martian basalts."},
    # --- Mars analogs ---
    {"id": "mdrs-utah", "name": "Mars Desert Research Station", "region": "Utah, USA", "body": "earth",
     "lat": 38.406, "lon": -110.792, "analog_for": ["mars"],
     "notes": "Long-running Mars simulation habitat in a sedimentary desert."},
    {"id": "rio-tinto", "name": "Río Tinto", "region": "Huelva, Spain", "body": "earth",
     "lat": 37.69, "lon": -6.59, "analog_for": ["mars"],
     "notes": "Acidic, iron-rich river system; chemistry analog for Martian sulfate deposits."},
    {"id": "pilbara-dresser", "name": "Pilbara, Dresser Formation", "region": "Western Australia", "body": "earth",
     "lat": -21.15, "lon": 119.42, "analog_for": ["mars"],
     "notes": "About 3.5 billion-year-old stromatolites; astrobiology analog for Jezero. Coordinates approximate."},
    {"id": "namib-sossusvlei", "name": "Sossusvlei dunes", "region": "Namib Desert, Namibia", "body": "earth",
     "lat": -24.73, "lon": 15.29, "analog_for": ["mars"],
     "notes": "Large active dune field; analog for Martian aeolian (wind) landforms."},
    # --- Steep polar terrain (test for lunar south pole crater walls) ---
    {"id": "vinson-massif", "name": "Vinson Massif", "region": "Ellsworth Mountains, Antarctica", "body": "earth",
     "lat": -78.525, "lon": -85.617, "analog_for": ["moon"],
     "notes": "Steep polar mountains with low sun angles. Team hypothesis: terrain test for Shackleton crater walls."},
]

TARGETS = [
    {"id": "jezero-crater", "name": "Jezero Crater", "region": "Mars · Perseverance (Mars 2020) landing site", "body": "mars",
     "lat": 18.38, "lon": 77.58, "notes": "Ancient lake basin with a river delta."},
    {"id": "gale-crater", "name": "Gale Crater", "region": "Mars · Curiosity (MSL) exploration area", "body": "mars",
     "lat": -5.4, "lon": 137.8, "notes": "Crater with a central layered mound (Aeolis Mons)."},
    {"id": "elysium-insight", "name": "Elysium Planitia", "region": "Mars · InSight landing site", "body": "mars",
     "lat": 4.502, "lon": 135.623, "notes": "Smooth volcanic plain."},
    {"id": "shackleton-crater", "name": "Shackleton Crater", "region": "Moon · South Pole", "body": "moon",
     "lat": -89.667, "lon": 129.783, "notes": "Polar crater; region considered for a permanent base near its rim."},
    {"id": "apollo-11-tranquillitatis", "name": "Mare Tranquillitatis", "region": "Moon · Apollo 11 landing site (equatorial)", "body": "moon",
     "lat": 0.674, "lon": 23.473, "notes": "Equatorial mare basalt plain."},
]

# Values from published papers. Example format (do NOT invent numbers):
# LITERATURE = {"gale-crater": {"temp_mean_c": {"value": -50.0, "citation": "Author et al. (Year), Journal, doi:..."}}}
LITERATURE = {}

WINDOW_KM = 20      # side of the square analysis window
PIXEL_M = 500       # common grid for ALL bodies (≈ Mars MOLA 463 m, the coarsest dataset)
MIN_COVERAGE = 0.95 # reject a window if less than 95% of it has valid data

ONLY = []           # e.g. ["mount-cameroon"] to run a single location while debugging
SHOW_PLOTS = True
SAVE_TO_DRIVE = True
DRIVE_OUT = "/content/drive/MyDrive/geoproxy/geoproxy-data/processed"

### 3 · Data sources (swap datasets here)
*Why:* every dataset is **one entry** in this table. To use a different dataset (e.g. a higher-resolution Mars DEM), add an entry and point `TERRAIN_SOURCE` at it; no other code changes. This table is also copied into the JSON, so the web app can show where every number comes from.

All of these are read **remotely, window by window**: nothing large is downloaded.

In [ ]:
BODIES = {
    "earth": {"radius_m": 6371000.0},
    "mars":  {"radius_m": 3396190.0},
    "moon":  {"radius_m": 1737400.0},
}

SOURCES = {
    "copernicus-glo90": {
        "body": "earth", "kind": "earth_tiles", "native_res_m": 90,
        "dataset": "Copernicus DEM GLO-90 (digital surface model, about 90 m)",
        "provider": "ESA / European Union Copernicus programme, via AWS Open Data",
        "url": "https://registry.opendata.aws/copernicus-dem/",
        "attribution": "© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA",
        "caveat": "Surface model: includes vegetation canopy and buildings, mostly smoothed out at 500 m.",
    },
    "mola-463m": {
        "body": "mars", "kind": "global_file", "native_res_m": 463,
        "path": "/vsicurl/https://asc-pds-services.s3.us-west-2.amazonaws.com/mosaic/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif",
        "dataset": "Mars MGS MOLA global DEM, 463 m",
        "provider": "NASA Mars Global Surveyor MOLA (PDS); global mosaic hosted by USGS Astrogeology",
        "url": "https://planetarymaps.usgs.gov/mosaic/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif",
        "attribution": "NASA / MOLA Science Team / USGS Astrogeology",
        "caveat": "Cells without laser tracks are interpolated; elevations are relative to the Mars areoid.",
    },
    "lola-118m": {
        "body": "moon", "kind": "global_file", "native_res_m": 118,
        "path": "/vsicurl/https://asc-pds-services.s3.us-west-2.amazonaws.com/mosaic/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif",
        "dataset": "Moon LRO LOLA global DEM, 118 m (March 2014)",
        "provider": "NASA Lunar Reconnaissance Orbiter LOLA (PDS); global mosaic hosted by USGS Astrogeology",
        "url": "https://planetarymaps.usgs.gov/mosaic/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif",
        "attribution": "NASA / LOLA Science Team / USGS Astrogeology",
        "caveat": "Elevations are relative to a 1737.4 km reference sphere.",
    },
    "nasa-power": {
        "body": "earth", "kind": "climate", "native_res_m": 50000,
        "dataset": "NASA POWER climatology (MERRA-2 based)",
        "provider": "NASA Langley Research Center POWER project",
        "url": "https://power.larc.nasa.gov/",
        "attribution": "NASA POWER Project",
        "caveat": "Coarse grid (about 50 km): describes the surrounding region, not a summit or valley floor.",
    },
}

TERRAIN_SOURCE = {"earth": "copernicus-glo90", "mars": "mola-463m", "moon": "lola-118m"}

FEATURES = {
    "relief_m":      {"label": "Relief", "unit": "m", "group": "terrain",
                      "definition": "90th minus 10th percentile of elevation in the window"},
    "slope_deg":     {"label": "Median slope", "unit": "degrees", "group": "terrain",
                      "definition": "Median slope of the window, from gradients of the 500 m grid"},
    "roughness_m":   {"label": "Roughness (TRI)", "unit": "m", "group": "terrain",
                      "definition": "Mean absolute elevation difference between each cell and its 8 neighbours (Terrain Ruggedness Index, Wilson et al. 2007)"},
    "temp_mean_c":   {"label": "Mean air temperature", "unit": "°C", "group": "environment",
                      "definition": "Annual mean temperature (Earth: NASA POWER T2M annual value)"},
    "temp_min_c":    {"label": "Minimum air temperature", "unit": "°C", "group": "environment",
                      "definition": "Low-temperature extreme (Earth: NASA POWER T2M_MIN annual value)"},
    "temp_max_c":    {"label": "Maximum air temperature", "unit": "°C", "group": "environment",
                      "definition": "High-temperature extreme (Earth: NASA POWER T2M_MAX annual value)"},
    "precip_mm_day": {"label": "Precipitation", "unit": "mm/day", "group": "environment",
                      "definition": "Mean precipitation (Earth: NASA POWER PRECTOTCORR annual value)"},
}

### 4 · Core functions (pure maths, same for every body)
- `aeqd_inverse`: converts metres on a flat map centred on the site into latitude/longitude on a sphere (azimuthal equidistant projection, Snyder 1987). This is what lets one piece of code work on Earth, Mars and the Moon, poles included.
- `window_points`: lays out the measurement grid, with several sub-samples per 500 m pixel so fine data is averaged, not just picked.
- `terrain_features`: computes relief, slope and roughness. **Absolute elevation is excluded**, because "zero" means sea level on Earth, the areoid on Mars and a sphere on the Moon.

In [ ]:
import math, os, time, traceback, json
import numpy as np
from scipy.ndimage import map_coordinates

def aeqd_inverse(x, y, lat0, lon0, radius):
    """Metres (x east, y north) on an azimuthal equidistant map centred at (lat0, lon0) -> (lat, lon) degrees."""
    phi0, lam0 = math.radians(lat0), math.radians(lon0)
    x = np.asarray(x, dtype="float64"); y = np.asarray(y, dtype="float64")
    rho = np.hypot(x, y)
    c = rho / radius
    safe_rho = np.where(rho > 0, rho, 1.0)
    lat = np.arcsin(np.clip(np.cos(c) * math.sin(phi0) + np.where(rho > 0, y * np.sin(c) * math.cos(phi0) / safe_rho, 0.0), -1, 1))
    lon = lam0 + np.arctan2(x * np.sin(c), rho * math.cos(phi0) * np.cos(c) - y * math.sin(phi0) * np.sin(c))
    return np.degrees(lat), (np.degrees(lon) + 180.0) % 360.0 - 180.0

def window_size_px():
    return int(round(WINDOW_KM * 1000 / PIXEL_M)) + 1   # odd, so the site sits in the centre pixel

def supersample_factor(native_res_m):
    return max(1, min(6, math.ceil(PIXEL_M / native_res_m)))

def window_points(lat, lon, body, k):
    """Lat/lon of every sub-sample point of the window (k x k points per output pixel). Row 0 = north."""
    n = window_size_px()
    half = n * PIXEL_M / 2
    offsets = -half + (np.arange(n * k) + 0.5) * (PIXEL_M / k)
    X, Y = np.meshgrid(offsets, offsets[::-1])
    return aeqd_inverse(X, Y, lat, lon, BODIES[body]["radius_m"])

def block_average(values, k):
    n = values.shape[0] // k
    return values.reshape(n, k, n, k).mean(axis=(1, 3))   # NaN anywhere in a block -> NaN pixel

def bilinear(data, rows, cols):
    return map_coordinates(data, [rows, cols], order=1, mode="nearest")

def terrain_features(z):
    coverage = float(np.isfinite(z).mean())
    if coverage < MIN_COVERAGE:
        return None, coverage
    gy, gx = np.gradient(z, PIXEL_M)
    slope = np.degrees(np.arctan(np.hypot(gx, gy)))
    c = z[1:-1, 1:-1]
    rows, cols = z.shape
    diffs = [np.abs(z[1 + dy:rows - 1 + dy, 1 + dx:cols - 1 + dx] - c)
             for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)]
    tri = np.mean(diffs, axis=0)
    return {
        "relief_m": float(np.nanpercentile(z, 90) - np.nanpercentile(z, 10)),
        "slope_deg": float(np.nanmedian(slope)),
        "roughness_m": float(np.nanmean(tri)),
    }, coverage

### 5 · Self-test (no internet needed)
*Why:* checks the maths on cases where we **know the right answer**. If this cell fails, the bug is in cell 4, not in the data. Always run it after changing cell 4.

In [ ]:
def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

R = BODIES["mars"]["radius_m"]; one_deg = R * math.radians(1)
lat, lon = aeqd_inverse([0.0], [0.0], 10, 20, R)
check("centre of the map is the site itself", abs(lat[0] - 10) < 1e-9 and abs(lon[0] - 20) < 1e-9)
lat, lon = aeqd_inverse([0.0], [one_deg], 10, 20, R)
check("1 degree of arc north -> latitude +1", abs(lat[0] - 11) < 1e-6 and abs(lon[0] - 20) < 1e-6)
lat, lon = aeqd_inverse([one_deg], [0.0], 0, 20, R)
check("1 degree of arc east at the equator -> longitude +1", abs(lat[0]) < 1e-6 and abs(lon[0] - 21) < 1e-6)
lat, lon = aeqd_inverse([0.0], [one_deg], -90, 0, R)
check("works at the south pole", abs(lat[0] + 89) < 1e-6)

n = window_size_px()
x = (np.arange(n) - n // 2) * PIXEL_M
X, Y = np.meshgrid(x, x)
f, cov = terrain_features(0.1 * X)                # a plane rising 10 m per 100 m eastwards
check("slope of a 10% plane = 5.71 degrees", abs(f["slope_deg"] - math.degrees(math.atan(0.1))) < 1e-6)
check("roughness of that plane = 37.5 m", abs(f["roughness_m"] - 37.5) < 1e-6)
check("flat ground has zero relief/slope/roughness", all(v == 0 for v in terrain_features(np.zeros((n, n)))[0].values()))
holed = np.zeros((n, n)); holed[:5, :] = np.nan
check("window with too much missing data is rejected", terrain_features(holed)[0] is None)

data = np.arange(12, dtype=float).reshape(3, 4)
check("bilinear interpolation", abs(bilinear(data, np.array([1.0]), np.array([1.5]))[0] - 5.5) < 1e-12)
check("block average", block_average(np.arange(16, dtype=float).reshape(4, 4), 2)[0, 0] == 2.5)
print("\nAll self-tests passed ✓")

### 6 · Data readers
*Why:* each reader takes a list of latitude/longitude points and returns elevations (in metres) at those points. It reads **only the part of the file that covers them**, over the internet.
- `earth_tiles`: Copernicus tiles are 1°×1° files named after their south-west corner. The needed tiles are stitched together.
- `global_file`: a single global map in equirectangular projection (Mars, Moon). Near a pole, a window spans all longitudes, so whole rows are read (a few tens of MB).
- `nasa_power`: one small web request per Earth site.

In [ ]:
import rasterio, requests
from rasterio.merge import merge
from rasterio.windows import Window

os.environ.update({
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",       # don't list remote folders
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES",       # fewer, larger web requests
    "GDAL_HTTP_MULTIRANGE": "YES",
    "GDAL_HTTP_MAX_RETRY": "4", "GDAL_HTTP_RETRY_DELAY": "2",
    "VSI_CACHE": "TRUE",
})

_OPEN = {}
def open_raster(path):
    if path not in _OPEN:
        _OPEN[path] = rasterio.open(path)
    return _OPEN[path]

def to_float(raw, nodata, scale=1.0, offset=0.0):
    a = raw.astype("float64")
    if nodata is not None and not np.isnan(nodata):
        a[a == nodata] = np.nan
    return a * (scale if scale is not None else 1.0) + (offset if offset is not None else 0.0)

def copernicus_tile(lat_floor, lon_floor):
    name = (f"Copernicus_DSM_COG_30_{'N' if lat_floor >= 0 else 'S'}{abs(lat_floor):02d}_00_"
            f"{'E' if lon_floor >= 0 else 'W'}{abs(lon_floor):03d}_00_DEM")
    return f"/vsicurl/https://copernicus-dem-90m.s3.amazonaws.com/{name}/{name}.tif"

def read_earth_tiles(source, lats, lons, log=print):
    west, east = lons.min() - 0.02, lons.max() + 0.02
    south, north = lats.min() - 0.02, lats.max() + 0.02
    srcs = []
    for a in range(math.floor(south), math.floor(north) + 1):
        for b in range(math.floor(west), math.floor(east) + 1):
            try:
                srcs.append(rasterio.open(copernicus_tile(a, b)))
            except Exception:
                log(f"   tile {a},{b} not available (open ocean or withheld)")
    if not srcs:
        raise RuntimeError("no Copernicus tiles cover this location")
    mosaic, tr = merge(srcs, bounds=(west, south, east, north), nodata=-9999)
    for s in srcs:
        s.close()
    data = to_float(mosaic[0], -9999)
    return bilinear(data, (lats - tr.f) / tr.e - 0.5, (lons - tr.c) / tr.a - 0.5)

def read_global_file(source, lats, lons, log=print):
    src = open_raster(source["path"])
    if src.crs.is_geographic:
        xs, ys = (lons + 180) % 360 - 180, lats
    else:
        p = src.crs.to_dict()
        if p.get("proj") != "eqc":
            raise ValueError(f"unsupported projection {p} (expected equirectangular 'eqc')")
        radius = p.get("R") or p.get("a") or BODIES[source["body"]]["radius_m"]
        lon0, lat0, lat_ts = p.get("lon_0", 0.0), p.get("lat_0", 0.0), p.get("lat_ts", 0.0)
        dlon = (lons - lon0 + 180) % 360 - 180
        xs = radius * math.cos(math.radians(lat_ts)) * np.radians(dlon) + p.get("x_0", 0.0)
        ys = radius * np.radians(lats - lat0) + p.get("y_0", 0.0)
    tr, H, W = src.transform, src.height, src.width
    cols = (xs - tr.c) / tr.a - 0.5
    rows = (ys - tr.f) / tr.e - 0.5
    r0 = max(int(np.floor(rows.min())) - 1, 0)
    r1 = min(int(np.ceil(rows.max())) + 2, H)
    if cols.max() - cols.min() > W / 2:          # window wraps around a pole or the date line
        c0, c1 = 0, W
        log(f"   polar/wrap-around window: reading {r1 - r0} full rows")
    else:
        c0 = max(int(np.floor(cols.min())) - 1, 0)
        c1 = min(int(np.ceil(cols.max())) + 2, W)
    raw = src.read(1, window=Window(c0, r0, c1 - c0, r1 - r0))
    data = to_float(raw, src.nodata, src.scales[0], src.offsets[0])
    return bilinear(data, rows - r0, cols - c0)

READERS = {"earth_tiles": read_earth_tiles, "global_file": read_global_file}

def read_points(source_key, lats, lons, log=print):
    source = SOURCES[source_key]
    flat = READERS[source["kind"]](source, np.ravel(lats), np.ravel(lons), log)
    return np.asarray(flat).reshape(np.shape(lats))

def nasa_power(lat, lon):
    r = requests.get("https://power.larc.nasa.gov/api/temporal/climatology/point",
                     params={"parameters": "T2M,T2M_MIN,T2M_MAX,PRECTOTCORR", "community": "AG",
                             "latitude": round(lat, 4), "longitude": round(lon, 4), "format": "JSON"},
                     timeout=90)
    r.raise_for_status()
    p = r.json()["properties"]["parameter"]
    value = lambda name: None if p[name]["ANN"] in (None, -999, -999.0) else float(p[name]["ANN"])
    return {"temp_mean_c": value("T2M"), "temp_min_c": value("T2M_MIN"),
            "temp_max_c": value("T2M_MAX"), "precip_mm_day": value("PRECTOTCORR")}

### 7 · Connection test: one known point per dataset
*Why:* proves each data source is reachable **and** read correctly, using places whose height is well known. If a number here is far off, the problem is the reader or the source, not your sites.

In [ ]:
KNOWN_POINTS = [
    ("copernicus-glo90", "Mount Cameroon summit area", 4.217, 9.167, "roughly 3,500-4,100 m"),
    ("mola-463m", "Olympus Mons summit", 18.65, -133.8, "roughly 21,000 m"),
    ("lola-118m", "Mare Tranquillitatis (Apollo 11)", 0.674, 23.473, "negative: a low mare plain, a few km below the reference sphere"),
]
for key, label, la, lo, expect in KNOWN_POINTS:
    t0 = time.time()
    try:
        v = read_points(key, np.array([la]), np.array([lo]))[0]
        print(f"OK    {key:17s} {label}: {v:,.0f} m (expected {expect})  [{time.time() - t0:.1f}s]")
    except Exception as e:
        print(f"FAIL  {key:17s} {label}: {type(e).__name__}: {e}")
for key in ("mola-463m", "lola-118m"):
    try:
        s = open_raster(SOURCES[key]["path"])
        print(f"      {key}: {s.width}x{s.height} px, nodata={s.nodata}, scale={s.scales[0]}, offset={s.offsets[0]}, crs={s.crs.to_dict()}")
    except Exception as e:
        print(f"      {key}: cannot open: {e}")
try:
    print("OK    nasa-power        Mount Cameroon area:", nasa_power(4.217, 9.167))
except Exception as e:
    print("FAIL  nasa-power:", type(e).__name__, e)

### 8 · Measure every location
*Why:* one loop for all sites and targets. Each location is wrapped in `try/except`: if one fails, **the error is printed and stored, and the others still run**. The table at the end is your checklist.

In [ ]:
import pandas as pd

locations = [dict(s, kind="site") for s in SITES] + [dict(t, kind="target") for t in TARGETS]
if ONLY:
    locations = [l for l in locations if l["id"] in ONLY]

results = []
for loc in locations:
    t0 = time.time()
    rec = {"loc": loc, "z": None, "terrain": None, "coverage": 0.0, "sea_fraction": None, "climate": None, "errors": []}
    print(f"\n▶ {loc['id']} ({loc['body']}, {loc['lat']}, {loc['lon']})")
    try:
        src_key = TERRAIN_SOURCE[loc["body"]]
        k = supersample_factor(SOURCES[src_key]["native_res_m"])
        lats, lons = window_points(loc["lat"], loc["lon"], loc["body"], k)
        rec["z"] = block_average(read_points(src_key, lats, lons), k)
        rec["terrain"], rec["coverage"] = terrain_features(rec["z"])
        valid = rec["z"][np.isfinite(rec["z"])]
        if loc["body"] == "earth" and valid.size:
            rec["sea_fraction"] = float(np.mean(valid <= 0))
        if rec["terrain"] is None:
            rec["errors"].append(f"terrain: coverage {rec['coverage']:.2f} below {MIN_COVERAGE}")
    except Exception as e:
        rec["errors"].append(f"terrain: {type(e).__name__}: {e}")
        traceback.print_exc()
    if loc["body"] == "earth":
        try:
            rec["climate"] = nasa_power(loc["lat"], loc["lon"])
        except Exception as e:
            rec["errors"].append(f"climate: {type(e).__name__}: {e}")
    rec["seconds"] = round(time.time() - t0, 1)
    print("  ", "✓" if not rec["errors"] else "✗", rec["terrain"], rec["climate"], rec["errors"] or "", f"[{rec['seconds']}s]")
    results.append(rec)

summary = pd.DataFrame([{
    "id": r["loc"]["id"], "body": r["loc"]["body"],
    **{k: (round(r["terrain"][k], 2) if r["terrain"] else None) for k in ("relief_m", "slope_deg", "roughness_m")},
    "temp_mean_c": (r["climate"] or {}).get("temp_mean_c"),
    "coverage": round(r["coverage"], 3), "sea": r["sea_fraction"], "s": r["seconds"],
    "errors": "; ".join(r["errors"]),
} for r in results])
summary

### 9 · Look at the windows
*Why:* always look at the data, not just the numbers. A wrong coordinate, a window in the sea or a strange-looking map shows up immediately.

In [ ]:
import matplotlib.pyplot as plt

shown = [r for r in results if r["z"] is not None]
if SHOW_PLOTS and shown:
    cols = 4
    rows_ = math.ceil(len(shown) / cols)
    fig, axes = plt.subplots(rows_, cols, figsize=(4.2 * cols, 3.6 * rows_), squeeze=False)
    e = WINDOW_KM / 2
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, shown):
        ax.axis("on")
        im = ax.imshow(r["z"], cmap="terrain", extent=[-e, e, -e, e])
        t = r["terrain"]
        ax.set_title(f"{r['loc']['name']} ({r['loc']['body']})\n" +
                     (f"relief {t['relief_m']:.0f} m · slope {t['slope_deg']:.1f}° · TRI {t['roughness_m']:.0f} m" if t else "no features"),
                     fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046, label="m")
    plt.tight_layout()
    plt.show()

### 10 · Build `sites.json` and `targets.json` (the data contract)
*Why:* the web app relies on this exact structure. Every feature value carries its unit, its **method** and its source:
- `measured`: computed by this pipeline from observation-based data (terrain)
- `modelled`: from a model or reanalysis product (NASA POWER climate)
- `literature`: from a published paper (with `citation`)
- `missing`: not available; the app must skip it, not treat it as zero

In [ ]:
from datetime import datetime, timezone

def feature_block(rec):
    loc, out = rec["loc"], {}
    for key, meta in FEATURES.items():
        val, method, source, citation = None, "missing", None, None
        if meta["group"] == "terrain" and rec["terrain"]:
            val, method, source = rec["terrain"][key], "measured", TERRAIN_SOURCE[loc["body"]]
        if meta["group"] == "environment" and rec["climate"] and rec["climate"].get(key) is not None:
            val, method, source = rec["climate"][key], "modelled", "nasa-power"
        lit = LITERATURE.get(loc["id"], {}).get(key)
        if val is None and lit:
            val, method, source, citation = lit["value"], "literature", "literature", lit["citation"]
        entry = {"value": None if val is None else round(float(val), 2), "unit": meta["unit"], "method": method, "source": source}
        if citation:
            entry["citation"] = citation
        out[key] = entry
    return out

def entry(rec):
    loc = rec["loc"]
    e = {k: loc[k] for k in ("id", "body", "name", "region", "lat", "lon") if k in loc}
    if loc["kind"] == "site":
        e["analog_for"] = loc.get("analog_for", [])
    e["notes"] = loc.get("notes", "")
    e["features"] = feature_block(rec)
    e["qc"] = {"coverage": round(rec["coverage"], 3), "sea_fraction": rec["sea_fraction"], "errors": rec["errors"]}
    return e

header = {
    "schema_version": "0.2",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "method": {"window_km": WINDOW_KM, "pixel_m": PIXEL_M, "min_coverage": MIN_COVERAGE,
               "projection": "azimuthal equidistant on each body's sphere, centred on each location",
               "resampling": "bilinear sub-samples averaged into each pixel",
               "method_values": ["measured", "modelled", "literature", "missing"]},
    "features": FEATURES,
    "sources": {k: {f: v for f, v in s.items() if f not in ("path", "kind")} for k, s in SOURCES.items()},
}
sites_json = {**header, "sites": [entry(r) for r in results if r["loc"]["kind"] == "site"]}
targets_json = {**header, "targets": [entry(r) for r in results if r["loc"]["kind"] == "target"]}
print(json.dumps(targets_json["targets"][:1], indent=2, ensure_ascii=False)[:3000])

### 11 · Save and download
*Why:* saving to Drive keeps the team's copy up to date. The downloads go into the repo at `data/sites.json` and `data/targets.json`.

In [ ]:
out_dir = "/content/output"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = DRIVE_OUT
    except Exception as e:
        print("Drive not available, saving locally:", e)
os.makedirs(out_dir, exist_ok=True)

for name, obj in (("sites.json", sites_json), ("targets.json", targets_json)):
    path = os.path.join(out_dir, name)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, ensure_ascii=False)
    print("Saved", path)
    try:
        from google.colab import files
        files.download(path)
    except Exception as e:
        print("   download skipped:", e)